# Extension: does the finding generalise? (2nd model + DoRA)

The original study found that on **Qwen2.5-1.5B / Alpaca**, LoRA makes *tiny (≈0.4–1.5%), high-rank, attention-biased edits that rescale rather than rotate representations* — but it was explicitly exploratory: one model, one PEFT variant.

This notebook turns that anecdote into a claim (or refutes it) along two axes:

1. **A second model family** — [SmolLM2-1.7B](https://huggingface.co/HuggingFaceTB/SmolLM2-1.7B) (ungated, Llama-style architecture).
2. **A second PEFT variant** — **DoRA** (Liu et al., 2024), which decomposes the update into magnitude and direction.

It also adds the **behavioural validation** the original study lacked: held-out instruction loss (base vs fine-tuned).

**Runtime**: 4 configs (2 models × LoRA/DoRA). ~40 min on an **A100/L4**, 2–3 h on a free T4. Reduce `MAX_SAMPLES` for a faster smoke run.

> ⚠️ **Cómo ejecutarlo**: celdas en orden, de arriba abajo. Todas las celdas son *idempotentes* (puedes re-ejecutar cualquiera sin romper nada). Si algo se queda en mal estado: *Entorno de ejecución → Reiniciar* y vuelve a ejecutar desde arriba.

In [ ]:
# Setup robusto e idempotente: siempre parte de /content, clona limpio y purga
# cualquier versión vieja del paquete que el kernel tenga cacheada.
import os, shutil, sys
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

REPO = "/content/Interpreting-LoRA-Fine-Tuning"

os.chdir("/content")
shutil.rmtree(REPO, ignore_errors=True)
!git clone -q https://github.com/mlahozy21/Interpreting-LoRA-Fine-Tuning.git
os.chdir(REPO)

!pip install -q -U "transformers>=4.44" "peft>=0.13" "datasets>=2.18" accelerate
!pip uninstall -y -q torchao   # el torchao preinstalado en Colab es incompatible con peft

# purgar módulos viejos y dejar solo la ruta absoluta del clon nuevo
for k in list(sys.modules):
    if k.startswith("lora_interp"):
        del sys.modules[k]
sys.path[:] = [p for p in sys.path if "Interpreting-LoRA" not in str(p)]
sys.path.insert(0, f"{REPO}/src")

# verificación: si esto falla, el código clonado no es el esperado
import inspect
from lora_interp.train import train_lora
assert "use_dora" in inspect.signature(train_lora).parameters, \
    "train.py sin use_dora: haz push del repo actualizado antes de continuar"
print("setup OK — cwd:", os.getcwd())
print("train_lora:", inspect.signature(train_lora))

In [ ]:
import copy, gc, json

import numpy as np
import torch

from lora_interp.analysis import adapter_update_norms, representation_drift
from lora_interp.train import train_lora, _format
from lora_interp.utils import set_seed, load_base

MODELS = ["Qwen/Qwen2.5-1.5B", "HuggingFaceTB/SmolLM2-1.7B"]
VARIANTS = [("lora", False), ("dora", True)]
RANK = 16
TARGET = "all"
MAX_SAMPLES = 2000          # reduce a 500 para una prueba rápida
SEED = 42

HELDOUT_DRIFT = [
    "### Instruction:\nExplain what overfitting is.\n\n### Response:\n",
    "### Instruction:\nSummarise the causes of the French Revolution.\n\n### Response:\n",
    "### Instruction:\nWrite a Python function that reverses a string.\n\n### Response:\n",
    "### Instruction:\nGive three tips for sleeping better.\n\n### Response:\n",
    "### Instruction:\nWhat is the capital of Australia?\n\n### Response:\n",
]

# batch segun la GPU: A100 (8,2) · L4 (4,4) · T4 (2,8) — mismo batch efectivo 16
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
BS, GA = (8, 2) if gpu_gb > 30 else (4, 4) if gpu_gb > 20 else (2, 8)
GC = gpu_gb < 30   # gradient checkpointing en T4/L4 (menos memoria, algo mas lento)
print(f"GPU: {gpu_gb:.0f} GB -> batch_size={BS}, grad_accum={GA}, grad_checkpoint={GC}")

print("device:", "cuda" if torch.cuda.is_available() else "cpu")

## Behavioural validation: held-out instruction loss

Mean token-level cross-entropy on 200 *held-out* Alpaca examples (never used in training — training takes the first `MAX_SAMPLES` of the seeded shuffle, we take the next 200).

In [ ]:
from datasets import load_dataset

_alpaca = load_dataset("tatsu-lab/alpaca", split="train").shuffle(seed=SEED)
EVAL_EXAMPLES = [_format(e) for e in _alpaca.select(range(MAX_SAMPLES, MAX_SAMPLES + 200))]


@torch.no_grad()
def heldout_loss(model, tok, texts, max_len=512):
    model.eval()
    losses = []
    for t in texts:
        ids = tok(t + tok.eos_token, return_tensors="pt", truncation=True,
                  max_length=max_len).input_ids.to(model.device)
        out = model(ids, labels=ids)
        losses.append(out.loss.item())
    return float(np.mean(losses))

print(f"{len(EVAL_EXAMPLES)} ejemplos held-out listos")

## Exact update norms — desde los factores LoRA, en fp32

`adapter_update_norms` calcula la actualización dirigida `dW = (alpha/r)·B@A` **en fp32, a partir de los factores LoRA**, y mide su norma relativa y su rango efectivo. Es de rango ≤ r por construcción, así que es directamente comparable con la Tabla 1 del estudio original.

> ⚠️ **No fusionamos el adaptador para medir el rango.** Hacer `merge_and_unload()` guarda el peso en bf16; restar ese peso fusionado del base recupera un `dW` contaminado por el redondeo de bf16 en todas las dimensiones, lo que **infla el rango efectivo de ~16 a varios cientos** (artefacto numérico, no un hallazgo). Para DoRA, `dW=(alpha/r)·B@A` es la componente **dirigida** (ignora el vector de magnitud), que es justo la cantidad de rango ≤ r que queremos comparar.

In [ ]:
# (El rango efectivo y la magnitud se calculan con adapter_update_norms
#  de src/lora_interp/analysis.py: dW=(alpha/r)*B@A en fp32, rango <= r.
#  Se evita merge_and_unload para no contaminar el rango con bf16.)

## Run the grid (2 models × LoRA/DoRA)

Idempotente: guarda cada configuración terminada en `results_partial.json`; si la celda se interrumpe y la relanzas, salta las configuraciones ya hechas.

In [ ]:
import os

ATTN_TYPES = {"q_proj", "k_proj", "v_proj", "o_proj"}
PARTIAL = "results_partial.json"
os.makedirs("figures", exist_ok=True)

results = json.load(open(PARTIAL)) if os.path.exists(PARTIAL) else []
done = {(r["model"], r["variant"]) for r in results}
base_losses = {}

for model_name in MODELS:
    short = model_name.split("/")[-1]
    for variant, use_dora in VARIANTS:
        if (short, variant) in done:
            print(f"{short}/{variant}: ya completado, salto")
            continue
        tag = f"{short}_{variant}_r{RANK}_{TARGET}"
        print(f"\n{'='*70}\n{tag}\n{'='*70}")
        model, tok = train_lora(model_name=model_name, rank=RANK, target=TARGET,
                                max_samples=MAX_SAMPLES, epochs=1.0,
                                output_dir=f"outputs/{tag}", seed=SEED,
                                use_dora=use_dora,
                                batch_size=BS, grad_accum=GA, grad_checkpoint=GC)

        # 1) loss base: mismo modelo con el adaptador desactivado (cero copias extra)
        if short not in base_losses:
            with model.disable_adapter():
                base_losses[short] = heldout_loss(model, tok, EVAL_EXAMPLES)
        # 2) validación conductual del fine-tuned
        ft_loss = heldout_loss(model, tok, EVAL_EXAMPLES)
        # 3) drift (adapter on/off) — antes del merge
        drift = representation_drift(model, tok, HELDOUT_DRIFT)
        # 4) magnitud + rango efectivo de la actualizacion dirigida dW=(alpha/r)*B@A,
        #    en fp32 desde los factores LoRA (rango <= r). NO se fusiona en bf16:
        #    el merge en bf16 infla el rango efectivo por ruido de redondeo.
        rows = adapter_update_norms(model)
        json.dump(rows, open(f"figures/update_norms_{tag}.json", "w"), indent=1)

        attn_u = [r["rel_update"] for r in rows if r["type"] in ATTN_TYPES]
        mlp_u = [r["rel_update"] for r in rows if r["type"] not in ATTN_TYPES]
        results.append({
            "model": short, "variant": variant,
            "rank": RANK, "target": TARGET,
            "heldout_loss_base": round(base_losses[short], 4),
            "heldout_loss_ft": round(ft_loss, 4),
            "mean_rel_update": round(float(np.mean([r["rel_update"] for r in rows])), 4),
            "rel_update_attn": round(float(np.mean(attn_u)), 4),
            "rel_update_mlp": round(float(np.mean(mlp_u)), 4),
            "mean_eff_rank": round(float(np.mean([r["effective_rank"] for r in rows])), 2),
            "drift_cos_final": round(drift["cosine_similarity"][-1], 4),
            "drift_relL2_final": round(drift["relative_l2"][-1], 4),
        })
        json.dump(results, open(PARTIAL, "w"), indent=1)   # checkpoint
        print(results[-1])

        # limpieza agresiva entre configuraciones
        del model, rows
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()
        print(f"GPU libre: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

print(f"\n{len(results)}/4 configuraciones completadas")


## Results

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df.to_csv("results_extension.csv", index=False)
print(df.to_markdown(index=False))

## What to check against the original four findings

1. **Tiny updates** — ¿`mean_rel_update` sigue ≲ 1–2% en SmolLM2?
2. **Full rank budget** — ¿`mean_eff_rank` ≈ r (16)? (medido desde BA en fp32; no desde el peso fusionado en bf16).
3. **Attention bias** — ¿`rel_update_attn > rel_update_mlp` en ambos modelos y variantes?
4. **Rescale, not rotate** — ¿`drift_cos_final` alto (≥0.92) con `drift_relL2_final` apreciable?

Más el chequeo conductual: `heldout_loss_ft < heldout_loss_base` en cada fila.

**Next step**: pega `results_extension.csv` como segunda tabla en el README bajo "Results" y actualiza el párrafo de *Limitations*.

> 💾 Antes de cerrar Colab, descarga `results_extension.csv`.